In [1]:
# Import libraries 
import pandas as pd
import numpy as np
import matplotlib
#matplotlib.use('Agg')          # remove this line if running in Jupyter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve,
                             accuracy_score, f1_score)
from sklearn.pipeline import Pipeline

In [2]:
# ── Colour palette ────────────────────────────────────────────────────────────
PALETTE = {
    "positive": "#E74C3C",
    "negative": "#2ECC71",
    "accent"  : "#3498DB",
    "dark"    : "#2C3E50",
    "mid"     : "#7F8C8D",
    "light"   : "#ECF0F1",
}


In [3]:
#  1. LOAD DATA

df = pd.read_csv('diabetes.csv')      # ← change path if needed
print("Shape:", df.shape)
print(df.head())
print(df.dtypes)

Shape: (768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64

In [7]:
#  2. MISSING VALUE DETECTION & IMPUTATION
#     Columns below cannot physically be zero → treat 0 as NaN
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# ── Figure 1 : Missing Value Analysis ──
zero_pct = {c: (df[c] == 0).sum() / len(df) * 100 for c in zero_cols}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor(PALETTE['dark'])
fig.suptitle("STEP 1 — Missing Value Analysis (Biological Zeros = NaN)",
             fontsize=16, fontweight='bold', color='white', y=0.98)

ax = axes[0, 0]
ax.set_facecolor('#1a252f')
bars = ax.barh(list(zero_pct.keys()), list(zero_pct.values()),
               color=[PALETTE['positive'] if v > 20 else PALETTE['accent']
                      for v in zero_pct.values()])
ax.set_xlabel('% Zero Values', color='white')
ax.set_title('Zero % per Column', color='white', fontweight='bold')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#34495e')
for bar, val in zip(bars, zero_pct.values()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', color='white', fontsize=9)
    
# Replace zeros with NaN
df_clean = df.copy()
for col in zero_cols:
    df_clean.loc[df_clean[col] == 0, col] = np.nan
    
    
# Impute with per-class median (smarter than global median)
for col in zero_cols:
    df_clean[col] = df_clean.groupby('Outcome')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Before / After distribution plots
for i, col in enumerate(['Glucose', 'Insulin', 'SkinThickness', 'BMI']):
    ax = axes[(i + 1) // 3, (i + 1) % 3]
    ax.set_facecolor('#1a252f')
    ax.hist(df[col], bins=30, alpha=0.5, color=PALETTE['mid'],
            label='Before', density=True)
    ax.hist(df_clean[col], bins=30, alpha=0.7, color=PALETTE['accent'],
            label='After impute', density=True)
    ax.set_title(f'{col} Before/After', color='white', fontweight='bold')
    ax.tick_params(colors='white')
    ax.legend(facecolor='#2c3e50', labelcolor='white', fontsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#34495e')

ax = axes[1, 2]
ax.set_facecolor('#1a252f')
miss_count = pd.DataFrame({
    'Column'     : zero_cols,
    'Zeros Fixed': [(df[c] == 0).sum() for c in zero_cols]
})
ax.bar(miss_count['Column'], miss_count['Zeros Fixed'], color=PALETTE['accent'])
ax.set_title('Records Fixed per Column', color='white', fontweight='bold')
ax.tick_params(colors='white', axis='both')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
for spine in ax.spines.values():
    spine.set_edgecolor('#34495e')

plt.tight_layout()
plt.savefig('fig1_missing_values.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['dark'])
plt.close()
print("✅  Fig 1 saved — missing values")

✅  Fig 1 saved — missing values


In [8]:
#  3. OUTLIER DETECTION & CAPPING  (IQR / Winsorisation)
# Outlier detection (IQR Method and capping)
num_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.patch.set_facecolor(PALETTE['dark'])
fig.suptitle("STEP 2 — Outlier Detection (IQR Method) & Capping",
             fontsize=16, fontweight='bold', color='white', y=0.98)

df_fixed = df_clean.copy()
outlier_counts = {}

for i, col in enumerate(num_cols):
    ax = axes[i // 4, i % 4]
    ax.set_facecolor('#1a252f')

    Q1  = df_fixed[col].quantile(0.25)
    Q3  = df_fixed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers            = ((df_fixed[col] < lower) | (df_fixed[col] > upper)).sum()
    outlier_counts[col] = outliers

    # Winsorise
    df_fixed[col] = df_fixed[col].clip(lower=lower, upper=upper)

    bp = ax.boxplot(
        [df_clean[col].dropna(), df_fixed[col]],
        patch_artist=True,
        boxprops    =dict(facecolor=PALETTE['accent'], alpha=0.7),
        medianprops =dict(color='yellow', linewidth=2),
        whiskerprops=dict(color='white'),
        capprops    =dict(color='white'),
        flierprops  =dict(marker='o', color=PALETTE['positive'], markersize=4),
    )
    bp['boxes'][1].set_facecolor(PALETTE['negative'])
    ax.set_xticklabels(['Before', 'After'], color='white')
    ax.set_title(f'{col}\n({outliers} outliers)', color='white',
                 fontweight='bold', fontsize=10)
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#34495e')

plt.tight_layout()
plt.savefig('fig2_outliers.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['dark'])
plt.close()
print("✅  Fig 2 saved — outliers")

✅  Fig 2 saved — outliers


In [9]:
#  4. EXPLORATORY DATA ANALYSIS (EDA)
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor(PALETTE['dark'])
fig.suptitle("STEP 3 — Exploratory Data Analysis",
             fontsize=18, fontweight='bold', color='white', y=0.98)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# Class balance pie
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_facecolor('#1a252f')
counts = df_fixed['Outcome'].value_counts()
ax0.pie(counts,
        labels     =['No Diabetes', 'Diabetes'],
        autopct    ='%1.1f%%',
        colors     =[PALETTE['negative'], PALETTE['positive']],
        textprops  ={'color': 'white'},
        startangle =90)
ax0.set_title('Class Distribution', color='white', fontweight='bold')

# Correlation heatmap
ax1 = fig.add_subplot(gs[0, 1:3])
ax1.set_facecolor('#1a252f')
corr = df_fixed.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax1, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
ax1.set_title('Correlation Matrix', color='white', fontweight='bold')
ax1.tick_params(colors='white', labelsize=8)

# Glucose distribution by Outcome
ax2 = fig.add_subplot(gs[0, 3])
ax2.set_facecolor('#1a252f')
for outcome, color, label in [
        (0, PALETTE['negative'], 'No Diabetes'),
        (1, PALETTE['positive'], 'Diabetes')]:
    ax2.hist(df_fixed[df_fixed['Outcome'] == outcome]['Glucose'],
             bins=25, alpha=0.7, color=color, label=label, density=True)
ax2.set_title('Glucose Distribution', color='white', fontweight='bold')
ax2.tick_params(colors='white')
ax2.legend(facecolor='#2c3e50', labelcolor='white', fontsize=8)
for spine in ax2.spines.values():
    spine.set_edgecolor('#34495e')

# Per-feature histograms by class
plot_cols = ['Glucose', 'BMI', 'Age', 'Insulin', 'BloodPressure', 'SkinThickness']
for i, col in enumerate(plot_cols):
    ax = fig.add_subplot(gs[1 + i // 4, i % 4])
    ax.set_facecolor('#1a252f')
    for outcome, color, label in [
            (0, PALETTE['negative'], 'No DM'),
            (1, PALETTE['positive'], 'DM')]:
        ax.hist(df_fixed[df_fixed['Outcome'] == outcome][col],
                bins=20, alpha=0.65, color=color, label=label, density=True)
    ax.set_title(col, color='white', fontweight='bold', fontsize=9)
    ax.tick_params(colors='white', labelsize=7)
    ax.legend(facecolor='#2c3e50', labelcolor='white', fontsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#34495e')

# Glucose vs BMI scatter
ax_sc = fig.add_subplot(gs[2, 2:4])
ax_sc.set_facecolor('#1a252f')
for outcome, color, label in [
        (0, PALETTE['negative'], 'No Diabetes'),
        (1, PALETTE['positive'], 'Diabetes')]:
    mask2 = df_fixed['Outcome'] == outcome
    ax_sc.scatter(df_fixed[mask2]['Glucose'], df_fixed[mask2]['BMI'],
                  alpha=0.5, c=color, s=20, label=label)
ax_sc.set_xlabel('Glucose', color='white')
ax_sc.set_ylabel('BMI', color='white')
ax_sc.set_title('Glucose vs BMI', color='white', fontweight='bold')
ax_sc.tick_params(colors='white')
ax_sc.legend(facecolor='#2c3e50', labelcolor='white', fontsize=8)
for spine in ax_sc.spines.values():
    spine.set_edgecolor('#34495e')

plt.savefig('fig3_eda.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['dark'])
plt.close()
print("✅  Fig 3 saved — EDA")

✅  Fig 3 saved — EDA


In [10]:
#  5. FEATURE ENGINEERING
df_feat = df_fixed.copy()

# Categorical bins
df_feat['Glucose_Risk'] = pd.cut(
    df_feat['Glucose'], bins=[0, 99, 125, 200], labels=[0, 1, 2]
).astype(int)

df_feat['BMI_Cat'] = pd.cut(
    df_feat['BMI'], bins=[0, 18.5, 24.9, 29.9, 100], labels=[0, 1, 2, 3]
).astype(int)

df_feat['Age_Group'] = pd.cut(
    df_feat['Age'], bins=[0, 30, 45, 60, 100], labels=[0, 1, 2, 3]
).astype(int)

# Interaction / ratio features
df_feat['Glucose_BMI']      = df_feat['Glucose'] * df_feat['BMI']
df_feat['Insulin_Glucose']  = df_feat['Insulin'] / (df_feat['Glucose'] + 1)
df_feat['Preg_Age']         = df_feat['Pregnancies'] / (df_feat['Age'] + 1)

print(f"Feature engineering done → {df_feat.shape[1]} total columns")

# ── Figure 4 : Feature Engineering Insights ──────────────────────────────────
new_feats   = ['Glucose_Risk', 'BMI_Cat', 'Age_Group',
               'Glucose_BMI', 'Insulin_Glucose', 'Preg_Age']
feat_titles = ['Glucose Risk Tier', 'BMI Category', 'Age Group',
               'Glucose × BMI', 'Insulin/Glucose Ratio', 'Pregnancies/Age']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.patch.set_facecolor(PALETTE['dark'])
fig.suptitle("STEP 4 — Feature Engineering",
             fontsize=16, fontweight='bold', color='white', y=0.98)

for i, (feat, title) in enumerate(zip(new_feats, feat_titles)):
    ax = axes[i // 3, i % 3]
    ax.set_facecolor('#1a252f')
    for outcome, color, label in [
            (0, PALETTE['negative'], 'No DM'),
            (1, PALETTE['positive'], 'DM')]:
        ax.hist(df_feat[df_feat['Outcome'] == outcome][feat],
                bins=20, alpha=0.7, color=color, label=label, density=True)
    ax.set_title(title, color='white', fontweight='bold')
    ax.tick_params(colors='white')
    ax.legend(facecolor='#2c3e50', labelcolor='white', fontsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#34495e')

plt.tight_layout()
plt.savefig('fig4_feature_eng.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['dark'])
plt.close()
print("✅  Fig 4 saved — feature engineering")


Feature engineering done → 15 total columns
✅  Fig 4 saved — feature engineering


6. MODEL TRAINING & EVALUATION

In [14]:
#  6. MODEL TRAINING & EVALUATION
feature_cols = [c for c in df_feat.columns if c != 'Outcome']
X = df_feat[feature_cols]
y = df_feat['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=8,
        random_state=42, class_weight='balanced'
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4,
        learning_rate=0.05, random_state=42
    ),
}

skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred    = model.predict(X_test)
    y_prob    = model.predict_proba(X_test)[:, 1]
    cv_scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')

    results[name] = {
        'model'   : model,
        'y_pred'  : y_pred,
        'y_prob'  : y_prob,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1'      : f1_score(y_test, y_pred),
        'roc_auc' : roc_auc_score(y_test, y_prob),
        'cv_mean' : cv_scores.mean(),
        'cv_std'  : cv_scores.std(),
        'report'  : classification_report(y_test, y_pred),
    }

    print(f"\n{'='*52}")
    print(f"  {name}")
    print(f"  Accuracy : {results[name]['accuracy']:.4f}")
    print(f"  F1 Score : {results[name]['f1']:.4f}")
    print(f"  ROC-AUC  : {results[name]['roc_auc']:.4f}")
    print(f"  CV AUC   : {results[name]['cv_mean']:.4f} ± {results[name]['cv_std']:.4f}")
    print(results[name]['report'])

best_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f"\n🏆 Best Model : {best_name}  "
      f"(ROC-AUC = {results[best_name]['roc_auc']:.4f})")

# ── Figure 5 : Model Evaluation ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.patch.set_facecolor(PALETTE['dark'])
fig.suptitle("STEP 5 — Model Training & Evaluation",
             fontsize=16, fontweight='bold', color='white', y=0.98)

model_names  = list(results.keys())
roc_colors   = [PALETTE['accent'], PALETTE['positive'], '#F39C12']

# Bar chart — metric comparison
ax = axes[0, 0]
ax.set_facecolor('#1a252f')
metrics = ['accuracy', 'f1', 'roc_auc']
x = np.arange(len(model_names))
width = 0.25
for j, (metric, color) in enumerate(
        zip(metrics, [PALETTE['accent'], PALETTE['negative'], PALETTE['positive']])):
    vals = [results[n][metric] for n in model_names]
    ax.bar(x + j * width, vals, width, label=metric.upper(), color=color, alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], color='white', fontsize=8)
ax.set_ylim(0.6, 1.0)
ax.set_title('Model Metrics Comparison', color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.legend(facecolor='#2c3e50', labelcolor='white', fontsize=8)
for spine in ax.spines.values():
    spine.set_edgecolor('#34495e')

# ROC curves
ax = axes[0, 1]
ax.set_facecolor('#1a252f')
for (name, res), color in zip(results.items(), roc_colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{name} (AUC={res['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
ax.set_xlabel('False Positive Rate', color='white')
ax.set_ylabel('True Positive Rate', color='white')
ax.set_title('ROC Curves', color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.legend(facecolor='#2c3e50', labelcolor='white', fontsize=8)
for spine in ax.spines.values():
    spine.set_edgecolor('#34495e')

# Confusion matrix — best model
ax = axes[0, 2]
ax.set_facecolor('#1a252f')
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No DM', 'DM'], yticklabels=['No DM', 'DM'],
            annot_kws={'size': 14, 'color': 'white'})
ax.set_title(f'Confusion Matrix\n({best_name})', color='white', fontweight='bold')
ax.tick_params(colors='white')

# Feature importance — Random Forest
ax = axes[1, 0]
ax.set_facecolor('#1a252f')
rf           = results['Random Forest']['model']
importances  = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
colors_imp   = [PALETTE['positive'] if v > importances.median() else PALETTE['accent']
                for v in importances]
ax.barh(importances.index, importances.values, color=colors_imp)
ax.set_title('Feature Importance (RF)', color='white', fontweight='bold')
ax.tick_params(colors='white', labelsize=8)
for spine in ax.spines.values():
    spine.set_edgecolor('#34495e')

# 5-Fold CV bar chart
ax = axes[1, 1]
ax.set_facecolor('#1a252f')
cv_means = [results[n]['cv_mean'] for n in model_names]
cv_stds  = [results[n]['cv_std']  for n in model_names]
ax.bar(range(len(model_names)), cv_means, yerr=cv_stds, capsize=8,
       color=roc_colors, alpha=0.85, error_kw={'color': 'white', 'linewidth': 2})
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], color='white', fontsize=8)
ax.set_ylim(0.6, 1.0)
ax.set_title('5-Fold CV ROC-AUC ± Std', color='white', fontweight='bold')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#34495e')

# Classification report heatmap — best model
ax = axes[1, 2]
ax.set_facecolor('#1a252f')
report    = classification_report(y_test, results[best_name]['y_pred'],
                                  target_names=['No DM', 'DM'], output_dict=True)
report_df = pd.DataFrame(report).T.iloc[:2][['precision', 'recall', 'f1-score']]
sns.heatmap(report_df, annot=True, fmt='.3f', cmap='YlGn', ax=ax,
            annot_kws={'size': 12}, vmin=0.5, vmax=1.0)
ax.set_title(f'Classification Report\n({best_name})', color='white', fontweight='bold')
ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig('fig5_model_eval.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['dark'])
plt.close()
print("✅  Fig 5 saved — model evaluation")
print("\n🎉 Pipeline complete! All figures saved.")


  Logistic Regression
  Accuracy : 0.7597
  F1 Score : 0.6606
  ROC-AUC  : 0.8441
  CV AUC   : 0.8777 ± 0.0191
              precision    recall  f1-score   support

           0       0.82      0.81      0.81       100
           1       0.65      0.67      0.66        54

    accuracy                           0.76       154
   macro avg       0.74      0.74      0.74       154
weighted avg       0.76      0.76      0.76       154


  Random Forest
  Accuracy : 0.8701
  F1 Score : 0.8113
  ROC-AUC  : 0.9370
  CV AUC   : 0.9394 ± 0.0121
              precision    recall  f1-score   support

           0       0.89      0.91      0.90       100
           1       0.83      0.80      0.81        54

    accuracy                           0.87       154
   macro avg       0.86      0.85      0.86       154
weighted avg       0.87      0.87      0.87       154


  Gradient Boosting
  Accuracy : 0.8701
  F1 Score : 0.8113
  ROC-AUC  : 0.9598
  CV AUC   : 0.9434 ± 0.0125
              prec